Bronze → Clean → SCD → History

1. Read Bronze

In [0]:
bronze = spark.read.table(
"opensky.bronze.bronze_flights"
)

In [0]:
bronze.printSchema()

Change Datatypes


In [0]:
from pyspark.sql.functions import expr, current_timestamp


silver_df = bronze.select(

    # Identifiers
    expr("try_cast(icao24 as string)").alias("icao24"),
    expr("try_cast(callsign as string)").alias("callsign"),
    expr("try_cast(origin_country as string)").alias("origin_country"),


    # Time fields
    expr("try_cast(time_position as long)").alias("time_position"),
    expr("try_cast(last_contact as long)").alias("last_contact"),


    # Location fields
    expr("try_cast(latitude as double)").alias("latitude"),
    expr("try_cast(longitude as double)").alias("longitude"),


    # Flight measurements
    expr("try_cast(baro_altitude as double)").alias("baro_altitude"),
    expr("try_cast(velocity as double)").alias("velocity"),
    expr("try_cast(true_track as double)").alias("true_track"),
    expr("try_cast(vertical_rate as double)").alias("vertical_rate"),
    expr("try_cast(geo_altitude as double)").alias("geo_altitude"),


    # Boolean
    expr("try_cast(on_ground as boolean)").alias("on_ground"),


    # Integer fields
    expr("try_cast(squawk as int)").alias("squawk"),
    expr("try_cast(position_source as int)").alias("position_source"),


    # Boolean flags
    expr("try_cast(spi as boolean)").alias("spi"),


    # Metadata
    expr("try_cast(ingestion_timestamp as timestamp)")
        .alias("ingestion_timestamp"),

    expr("try_cast(load_date as date)")
        .alias("load_date"),


    expr("source_file").alias("source_file")

)

In [0]:
silver_df.printSchema()

drop unwanted columns

In [0]:
silver_df = silver_df.drop(
    "sensors",
    "geo_altitude",
    "squawk",
    "spi",
    "position_source",
    "_rescued_data"
)

Remove duplicates:

In [0]:
from pyspark.sql.functions import *


silver_df = (

silver_df

.dropDuplicates(
[
"icao24",
"last_contact"
]
)

)

Remove bad data:

In [0]:
silver_df = silver_df.filter(

(col("latitude").between(-90,90))

)

Add Flight Status

In [0]:
from pyspark.sql.functions import *

silver_df = bronze.withColumn(
    "flight_status",
    when(col("on_ground") == True, "Ground")
    .otherwise("Flying")
)

Add Speed Category

In [0]:
from pyspark.sql.functions import when, col


silver_df = silver_df.withColumn(
    "speed_category",

    when(
        col("velocity") <= 100,
        "Slow"
    )

    .when(
        col("velocity") <= 500,
        "Normal"
    )

    .otherwise(
        "Fast"
    )
)

Add Altitude Category

In [0]:
silver_df = silver_df.withColumn(
    "altitude_category",
    when(col("baro_altitude") < 1000,"Low")
    .when(col("baro_altitude") < 10000,"Medium")
    .otherwise("High")
)

Add Direction Category

In [0]:
silver_df = silver_df.withColumn(
    "direction",
    when(col("true_track").between(0,90),"North-East")
    .when(col("true_track").between(90,180),"South-East")
    .when(col("true_track").between(180,270),"South-West")
    .otherwise("North-West")
)

Add Silver Timestamp

In [0]:
silver_df = silver_df.withColumn(

"silver_timestamp",

current_timestamp()

)

Write Silver Table

In [0]:
%sql
DROP TABLE IF EXISTS opensky.silver.silver_flights;

In [0]:
silver_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"opensky.silver.silver_flights"
)

New Bronze Data
        |
        ↓
Compare with Silver
        |
        ↓
Existing aircraft?
        |
    YES → UPDATE
        |
    NO → INSERT

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp


def upsert_scd(batch_df, batch_id):

    if batch_df.count() == 0:
        return

    target = DeltaTable.forName(
        spark,
        "opensky.silver.aircraft_history"
    )


    (
        target.alias("target")
        .merge(
            batch_df.alias("source"),
            """
            target.icao24 = source.icao24
            AND target.is_current = true
            """
        )

        .whenMatchedUpdate(
            condition="""
            target.country <> source.origin_country
            """,
            set={
                "is_current": "false",
                "end_date": "current_timestamp()"
            }
        )

        .whenNotMatchedInsert(
            values={
                "icao24": "source.icao24",
                "country": "source.origin_country",
                "start_date": "current_timestamp()",
                "end_date": "NULL",
                "is_current": "true"
            }
        )

        .execute()
    )

SCD TYPE 2 HISTORY TRACKING

Create history table:

Example:

Before:

icao24	country	current
abc123	USA	true

After:

icao24	country	current
abc123	USA	false
abc123	Canada	true

In [0]:
%sql 
Drop table opensky.silver.aircraft_history

In [0]:
%sql
CREATE TABLE opensky.silver.aircraft_history
(
    icao24 STRING,
    country STRING,
    start_date TIMESTAMP,
    end_date TIMESTAMP,
    is_current BOOLEAN
)
USING DELTA;

In [0]:
silver_df.printSchema()